<a href="https://colab.research.google.com/github/adamsiehen/antyspamm/blob/main/Adam_Siehe%C5%84_informatyka_stosowana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Klasyfikacja e-maili na spam i niespam.

Autor: Adam Sieheń


## 1. Pobranie danych z bazy kaggle

Link do datasetu: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/sms-spam-collection-dataset


## 2. Zaimportowanie biblioteki pandas i wczytanie pliku csv do zmiennej df

In [2]:
import pandas as pd

# Wczytaj plik CSV
df = pd.read_csv(path + "/spam.csv", encoding='latin-1')

# Podgląd pierwszych wierszy
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


## 3. Wczytanie i oczyszczenie danych

In [3]:
# Zostawiamy tylko dwie potrzebne kolumny
df = df[['v1', 'v2']]
df.columns = ['label', 'message']  # zmiana nazw kolumn

# Sprawdzamy rozkład klas
print(df['label'].value_counts())

# Zamieniamy etykiety na liczby: ham = 0, spam = 1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

label
ham     4825
spam     747
Name: count, dtype: int64


<ipython-input-3-5d5e52f3e70d>:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})


## 4. Wektoryzacja tekstu i trening modelu (Naive Bayes)

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

# Podział na dane treningowe i testowe
X_train, X_test, y_train, y_test = train_test_split(df['message'], df['label_num'], test_size=0.2, random_state=42)

# Wektoryzacja tekstu - przekształcenie wiadomości na wektory TF-IDF
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Trening modelu Naive Bayes
model = MultinomialNB()
model.fit(X_train_vec, y_train)

# Predykcja
y_pred = model.predict(X_test_vec)

# Ewaluacja
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Ham", "Spam"]))

[[965   0]
 [ 37 113]]
              precision    recall  f1-score   support

         Ham       0.96      1.00      0.98       965
        Spam       1.00      0.75      0.86       150

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.96      1115



## 5. Interpretacja
### 5.1 Macierz pomyłek
[[965   0]

 [ 37 113]]

To macierz pomyłek

965 wiadomości "ham", czyli niespamm zostało poprawnie zaklasyfikowanych jako niespam.

113 wiadomości "spam" poprawnie rozpoznane jako spam.

37 wiadomości spam błędnie oznaczone jako niespam — to false negatives.

### 5.2 Miary precyzji
Precision (spam = 1.00): Gdy model mówi "to spam", ma 100% racji – nie popełnia false positives.

Recall (spam = 0.75): Ale wykrywa tylko 75% wszystkich spamów – przegapia co czwartego spama.

F1-score (spam = 0.86): Średnia z precision i recall

Accuracy = 0.97

Czyli: 97% wszystkich wiadomości zostało poprawnie sklasyfikowanych.

6. Przykład użycia

In [14]:
def classify_message(msg):
    msg_vec = vectorizer.transform([msg])
    prediction = model.predict(msg_vec)[0]
    return "Spam" if prediction == 1 else "Ham"

# Wiadomość, która może być spammem ale niekoniecznie, klasyfikacja OK:
print(classify_message("Congratulations! You have won a free ticket."))

Ham


In [18]:
# Wiadomość, która napewno nie jest spammem, klasyfikacja OK:
print(classify_message("Hello, please call me"))

Ham


In [16]:
# Ewidentny przykład spammu, klasyfikacja błędna
print(classify_message("ACTION REQUIRED. Please verify your Bank of America account information to avoid a hold on your account. Click here to confirm: https://blog.textingbase.com/how-to-identify-spam-text-messages"))

Ham


In [19]:
# Ewidentny spamm, klasyfikacja poprawna
print(classify_message("WINNER!! As a valued network customer you have been selected to receivea �900 prize reward!"))

Spam


# Podsumowanie

Model nie zna słów typu "verify", "account", "click here" jako spamowych, bo model był trenowany na SMS-ach, nie wiadomościach e-mail/finansowych.

Link w wiadomości jest unikalny – model nie widział wcześniej podobnych treści, więc nie przypisał ich do spamu.

Brak wzorców powtarzających się w treści – TF-IDF traktuje każde słowo niezależnie i rzadkie słowa nie mają wpływu, jeśli nie występowały w treningu.